In [ ]:
# Core
import os
import pandas as pd
import copy
import numpy as np
import pybedtools

# Graphs
import matplotlib.pyplot as plt
import seaborn as sns
# Statistics
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
from scipy.stats import mannwhitneyu
from itertools import combinations
from scipy.stats import binomtest
from adjustText import adjust_text
from sklearn.metrics import precision_recall_curve
from scipy.stats import spearmanr
from scipy.stats import pearsonr

from scipy.stats import ks_2samp # Kolmogorov–Smirnov (KS)
from scipy.stats import linregress
from scipy.stats import gaussian_kde


In [ ]:
import os, sys
# --- Shared helpers: const.py lives in analyses/common ---
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'common')))
import const
from const import pos_active_ctrl_color,neg_active_ctrl_color,highlight_color,custom_cmap
from const import set_equal_plot_limits
from const import plot_color_pallete
from const import custom_cmap_bolder
from const import FONT_SIZE_small
from const import save_fig
const.set_plot_style()

# --- Base directory (EDIT THIS) ---
# Paths below are relative to this lab-data root (the Collaboration directory).
BASE_DIR = '/home/labs/davidgo/Collaboration'   # <- set to your local copy
os.chdir(BASE_DIR)

In [ ]:
# Parameters
filter_for_oligos_w_1_variant = False
fig_dir ='humanMPRA/TF_analysis/final/figures'
dataset_dir = 'humanMPRA/TF_analysis/final/datasets_for_paper'

In [ ]:
# Load MPRA results
MPRA_output= pd.read_csv(f'/home/labs/davidgo/Collaboration/humanMPRA/top_candidates/chondrocytes/humanMPRA_annotations_v3.csv', 
                     header=0)

MPRA_output.rename(columns={'oligo':'seq_id',}, inplace=True)

# Add fields
MPRA_output['active_bool'] = MPRA_output['differential_activity'].isin([True, False])
MPRA_output['diff_active_bool'] = MPRA_output['differential_activity']==True

#keep only relevant fields
MPRA_output = MPRA_output[['seq_id','variants_genomic','variants_count','logFC_derived_vs_ancestral','activity_fdr_ancestral',
                           'activity_fdr_derived','differential_activity_fdr','active_bool','diff_active_bool']]


if filter_for_oligos_w_1_variant:
    print(f"Filtering for oligos with exactly 1 variant. Original count: {len(MPRA_output)}, Filtered count: {len(MPRA_output[MPRA_output['variants_count'] == 1])}")
    MPRA_output = MPRA_output[MPRA_output['variants_count'] == 1]


In [ ]:
# Load FIMO and PBM results
hMPRA_PBM = pd.read_csv(f'humanMPRA/TF_analysis/final/results/hMPRA_PBM_oligo_TF_merged.tsv', sep='\t')
hMPRA_PBM['motif_id_clean'] = hMPRA_PBM['motif_id_clean'].str.upper()

hMPRA_FIMO = pd.read_csv(f'humanMPRA/TF_analysis/final/results/hMPRA_FIMO_oligo_TF_merged.tsv', sep='\t')
hMPRA_FIMO['motif_id_clean'] = hMPRA_FIMO['motif_id_clean'].str.upper()


## Neither is in PBM

In [ ]:
TFs_of_interest = ['ZFP42'] # Alternatives


In [ ]:
path = "/home/labs/davidgo/Collaboration/USEFUL_DATASETS/Sequence/Motifs_and_TFBS/PBM/data/pbm_8mer_data.pkl"
pbm_data = pd.read_pickle(path)

TFs_of_interest = ['ZFAT', 'ZFHX4', 'ZNF18'] # The original ones
TFs_of_interest = ['ZFP42'] # Alternatives


found = any(
    s.lower() in key.lower()
    for key in pbm_data
    for s in TFs_of_interest
)

print(found)

In [ ]:
jaspar_metadata = pd.read_csv('USEFUL_DATASETS/Sequence/Motifs_and_TFBS/JASPAR2024/JASPAR2024_CORE_vertebrates_non-redundant_meta.tsv', 
                     header=0,sep = '\t')
jaspar_metadata = jaspar_metadata.rename(columns={'Unnamed: 0': 'Matrix_ID'})

jaspar_metadata['TF'] = jaspar_metadata['TF'].str.upper()

In [ ]:
#check if TFs of interest are in JASPAR
found_in_jaspar = any(
    s in jaspar_metadata['TF'].values
    for s in TFs_of_interest        
)

print(found_in_jaspar)

In [ ]:
TFs_of_interest

In [ ]:
jaspar_metadata[jaspar_metadata['Matrix_ID']=='MA2590.1']

In [ ]:
jaspar_metadata

In [ ]:
all_motifs = pd.concat([hMPRA_PBM, hMPRA_FIMO], axis = 0)

In [ ]:
merged_df = pd.merge(all_motifs, MPRA_output, on='seq_id', how='outer')
PBM_merged_df = pd.merge(hMPRA_PBM, MPRA_output, on='seq_id', how='outer')
FIMO_merged_df = pd.merge(hMPRA_FIMO, MPRA_output, on='seq_id', how='outer')

In [ ]:
FIMO_TFs = hMPRA_FIMO['motif_id_clean'].unique()
PBM_TFs = hMPRA_PBM['motif_id_clean'].unique()

In [ ]:
TFs_of_interest = ['ZFAT', 'ZFHX4', 'ZNF18']
for tf in TFs_of_interest:
    if tf in FIMO_TFs:
        print(f"{tf} is in FIMO_TFs")
    else:
        print(f"{tf} is NOT in FIMO_TFs")

for tf in TFs_of_interest:
    if tf in PBM_TFs:
        print(f"{tf} is in PBM_TFs")
    else:
        print(f"{tf} is NOT in PBM_TFs")

In [ ]:
gene_annotation_table = pd.read_csv('/home/labs/davidgo/nadavmi/usefull/Human.GRCh38.p13.annot.tsv', 
                     header=0,sep = '\t',usecols=[0,1])


articular_cartilage_TPM = pd.read_csv('/home/labs/davidgo/Collaboration/USEFUL_DATASETS/Expression/Cartilage/Human/GSE114007_human_control_and_osteoarithritis_cartilage/GSE114007_norm_counts_TPM_GRCh38.p13_NCBI.tsv', 
                     header=0,sep = '\t')

articular_cartilage_TPM = articular_cartilage_TPM.iloc[:,0:9]
articular_cartilage_TPM['articular_cartilage_mean'] = articular_cartilage_TPM.iloc[:, 1:].mean(axis=1)
articular_cartilage_TPM = articular_cartilage_TPM[['GeneID','articular_cartilage_mean']]


articular_cartilage_TPM = pd.merge(articular_cartilage_TPM, gene_annotation_table, on='GeneID', how='outer') 
articular_cartilage_TPM = articular_cartilage_TPM.set_index('Symbol')
articular_cartilage_TPM = articular_cartilage_TPM[['articular_cartilage_mean']]
articular_cartilage_TPM = articular_cartilage_TPM.groupby('Symbol', as_index=True).mean() # Group by gene name and average duplicates


In [ ]:
articular_cartilage_TPM[articular_cartilage_TPM.index.isin(['ZFP42'])]

In [ ]:
articular_cartilage_TPM[articular_cartilage_TPM.index.isin(['ZFAT', 'ZFHX4', 'ZNF18'])]

## Comparing positive controls in hMPRA chonds to original oligos in mhMPRA  (osteoblats only). Check (1) Similar direction and (2) correlation.

In [ ]:
mhMPRA_dir = "/home/labs/davidgo/Collaboration/USEFUL_DATASETS/Expression/MPRAs/Modern human derived MPRA/comparative_df_Hob.csv"
mhMPRA_df = pd.read_csv(mhMPRA_dir, header=0)
mhMPRA_df["seq"] = mhMPRA_df['Sequence ID'].str.extract(r"(seq\d+)", expand=False)
mhMPRA_df = mhMPRA_df[['Sequence ID', 'seq', 'logFC','differentialy_active']]


In [ ]:
libraries = [
    #"L3a3",
     "L1a1", "L1a2", "L1a3",
     "L2a1", "L2a2", "L2a3",
     "L3a1", "L3a2", "L3a3",
     "L4a1",
]

base_dir = "/home/labs/davidgo/Collaboration/humanMPRA/chondrocytes"
rel_path = "output/mpranalyze_comparative/posctrl/mpranalyze_comp_res_filter_sorted.txt"

dfs = []
percents = []

for lib in libraries:
    positive_controls_dir = os.path.join(base_dir, lib, rel_path)
    df = pd.read_csv(positive_controls_dir, sep="\t")
    df["library"] = lib
    print(lib)

    successful_controls = (df["fdr"] < 0.05).sum()
    total_controls = len(df)

    pct = (successful_controls / total_controls) * 100 if total_controls else float("nan")
    percents.append(pct)

    dfs.append(df)

positive_controls = pd.concat(dfs, ignore_index=False)

In [ ]:
positive_controls_ost = positive_controls[positive_controls.index.str.contains("ost", case=False, na=False)].copy()

positive_controls_ost["seq"] = positive_controls_ost.index.to_series().str.extract(r"(seq\d+)", expand=False)

positive_controls_ost = positive_controls_ost[["seq", "logFC", "fdr", "library"]].copy()

In [ ]:
positive_controls_ost

In [ ]:

# merge
merged = positive_controls_ost.merge(
    mhMPRA_df,
    on="seq",
    how="inner"
)

print(len(merged))
merged = merged[merged['fdr'] < 0.05]
print(len(merged))


merged_collapsed = (
    merged
    .groupby("seq", as_index=False)
    .agg(
        logFC_x=("logFC_x", "mean"),
        logFC_y=("logFC_y", "mean"),
        fdr=("fdr", "mean"),
        library=("library", lambda x: "; ".join(x.dropna().astype(str).unique()))
    )
)
#change col names
merged_collapsed = merged_collapsed.rename(columns={"logFC_x": "logFC_hMPRA", "logFC_y": "logFC_mhMPRA"})

In [ ]:
len(merged['seq'].unique())

In [ ]:
merged_collapsed

In [ ]:

import matplotlib.pyplot as plt
from scipy.stats import pearsonr

x = merged_collapsed["logFC_hMPRA"]
y = merged_collapsed["logFC_mhMPRA"]

# Remove rows with missing values in either column
valid = x.notna() & y.notna()
x = x[valid]
y = y[valid]

# Pearson correlation
r, p = pearsonr(x, y)

fig, ax = plt.subplots(figsize=(6, 6))

ax.scatter(
    x,
    y,
    alpha=0.7
)

# Dashed reference lines at x = 0 and y = 0
ax.axhline(0, linestyle="--", linewidth=1)
ax.axvline(0, linestyle="--", linewidth=1)

ax.set_xlabel("logFC_hMPRA")
ax.set_ylabel("logFC_mhMPRA")

# Add Pearson r and p-value to plot
ax.text(
    0.05, 0.95,
    f"Pearson r = {r:.3f}\nP = {p:.2e}\nN = {len(x)}",
    transform=ax.transAxes,
    va="top",
    ha="left"
)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

x_col = "logFC_hMPRA"
y_col = "logFC_mhMPRA"

direction_df = merged_collapsed[[x_col, y_col]].dropna().copy()

# Direction aligns if both values have the same sign:
# both positive or both negative
direction_df["direction_aligns"] = np.sign(direction_df[x_col]) == np.sign(direction_df[y_col])

# Optional: exclude cases where one or both values are exactly 0
direction_df = direction_df[
    (direction_df[x_col] != 0) & 
    (direction_df[y_col] != 0)
].copy()

n_total = len(direction_df)
n_aligned = direction_df["direction_aligns"].sum()
percent_aligned = 100 * n_aligned / n_total

print(f"Aligned direction: {n_aligned}/{n_total}")
print(f"Percent aligned: {percent_aligned:.2f}%")